In [1]:
!pip uninstall -y -q bitsandbytes
!pip install -q --upgrade --no-cache-dir "transformers>=4.44.0" "accelerate>=0.34.0" "peft>=0.12.0" "datasets>=2.19.0" "trl>=0.9.6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 240.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 393.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 401.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 361.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 398.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 154.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 301.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 321.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 375.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 304.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 263.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List

import pandas as pd
import ast
from sklearn.model_selection import train_test_split

import torch

import peft
import transformers
from datasets import Dataset, DatasetDict
from peft import (
    LoraConfig, 
    get_peft_model 
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments
)

print(f"Transformers: {transformers.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"Torch: {torch.__version__}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

2025-12-22 10:36:15.916377: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766399776.135201      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766399776.198430      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Transformers: 4.57.3
PEFT: 0.18.0
Torch: 2.6.0+cu124
GPU: Tesla T4


In [ ]:
DATA_PATH = "/kaggle/input/new-roles-datasets/Cleaned/instructions.csv"

df = pd.read_csv(DATA_PATH)

lower = {c.lower(): c for c in df.columns}
text_key = "text" if "text" in lower else ("clean_data" if "clean_data" in lower else None)
label_key = "role" if "role" in lower else ("label" if "label" in lower else None)
opts_key  = "options" if "options" in lower else None

assert text_key is not None, f"CSV must contain a text column (one of: text, clean_data). Found: {list(df.columns)}"
assert label_key is not None, f"CSV must contain a label column (one of: role, label). Found: {list(df.columns)}"

df = df.rename(columns={
    lower[text_key]: "text",
    lower[label_key]: "label",
    **({lower[opts_key]: "options"} if opts_key else {})
})

df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
df["text"]  = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip()

global_options = sorted(df["label"].unique().tolist())

def parse_options(x):
    """
    Accepts:
      - JSON list string: '["A","B","C"]'
      - Python list literal: "['A','B']"
      - Pipe-separated: "A|B|C"
      - Comma-separated: "A,B,C"
    Returns a clean list[str] or None.
    """
    if pd.isna(x):
        return None
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return None
        try:
            val = ast.literal_eval(s)
            if isinstance(val, (list, tuple)):
                return [str(t).strip() for t in val if str(t).strip()]
        except Exception:
            pass
        sep = "|" if "|" in s else ("," if "," in s else None)
        if sep:
            return [t.strip() for t in s.split(sep) if t.strip()]
    return None

if "options" in df.columns:
    df["options"] = df["options"].apply(parse_options)
else:
    df["options"] = None

def ensure_contains_label(opts, gold):
    opts = opts or global_options
    if gold not in opts:
        opts = [gold] + [o for o in opts if o != gold]
    return opts

df["options"] = [ensure_contains_label(o, y) for o, y in zip(df["options"], df["label"])]

In [4]:
train_df, vt = train_test_split(
    df, test_size=0.2, random_state=42, shuffle=True,
    stratify=df["label"] if df["label"].nunique() > 1 else None
)

val_df, test_df = train_test_split(
    vt, test_size=0.5, random_state=42, shuffle=True,
    stratify=vt["label"] if vt["label"].nunique() > 1 else None
)


ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
})
ds

DatasetDict({
    train: Dataset({
        features: ['label', 'text', 'options'],
        num_rows: 15159
    })
    validation: Dataset({
        features: ['label', 'text', 'options'],
        num_rows: 1895
    })
})

In [6]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
dtype = torch.bfloat16 if use_bf16 else torch.float16
print("Using dtype:", dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

assert tokenizer is not None, "Failed to load tokenizer/processor."
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False

Using dtype: torch.float16


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [8]:
SYSTEM_PROMPT = (
    "You are a careful classifier. Choose exactly ONE label from the provided options. "
    "Return ONLY the label text with no extra words."
)

def build_messages(text: str, options: List[str], answer: str | None = None):
    opts_str = ", ".join(options)
    user_msg = (
        "Classify the following text by choosing exactly ONE label from the options.\n\n"
        f"Options: {opts_str}\n\n"
        f'Text:\n"""{text}"""\n\n'
        "Return ONLY one label from the Options."
    )
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    if answer is not None:
        msgs.append({"role": "assistant", "content": answer})
    return msgs

def _encode_chat(tokenizer, messages: List[Dict[str, Any]], add_gen_prompt: bool):
    """
    Robustly get a 1D torch.LongTensor of token IDs from apply_chat_template.
    """
    out = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=add_gen_prompt,
        tokenize=True,
        return_tensors=None
    )
    ids = out["input_ids"] if isinstance(out, dict) else out
    if isinstance(ids, torch.Tensor):
        ids = ids.squeeze(0).tolist()
    return torch.tensor(ids, dtype=torch.long)



In [9]:
@dataclass
class OptionsCollator:
    tokenizer: Any
    max_length: int = 320

    def __call__(self, batch: List[Dict[str, Any]]):
        pad_id = self.tokenizer.pad_token_id or self.tokenizer.eos_token_id
        input_ids_list, labels_list, attn_mask_list = [], [], []

        for ex in batch:
            text = str(ex["text"])
            gold = str(ex["label"])
            options = list(ex["options"]) if isinstance(ex["options"], (list, tuple)) else list(global_options)

            prompt_ids = _encode_chat(self.tokenizer, build_messages(text, options, None), add_gen_prompt=True)
            full_ids   = _encode_chat(self.tokenizer, build_messages(text, options, gold), add_gen_prompt=False)

            labels = full_ids.clone()
            labels[: len(prompt_ids)] = -100

            if full_ids.size(0) > self.max_length:
                cut = full_ids.size(0) - self.max_length
                full_ids = full_ids[cut:]
                labels   = labels[cut:]

            attn_mask = torch.ones_like(full_ids)
            input_ids_list.append(full_ids)
            labels_list.append(labels)
            attn_mask_list.append(attn_mask)

        max_len = max(x.size(0) for x in input_ids_list)

        def pad_to(t):
            if t.size(0) < max_len:
                pad_len = max_len - t.size(0)
                return torch.cat([t, torch.full((pad_len,), pad_id, dtype=t.dtype)], dim=0)
            return t

        input_ids = torch.stack([pad_to(x) for x in input_ids_list])
        labels    = torch.stack([pad_to(x) for x in labels_list])
        attention = torch.stack([pad_to(x) for x in attn_mask_list])

        labels[input_ids == pad_id] = -100
        return {"input_ids": input_ids, "attention_mask": attention, "labels": labels}

collator = OptionsCollator(tokenizer, max_length=320)

In [10]:
OUTPUT_DIR = "/kaggle/working/qwen2.5-1_5b-gencls-lora"

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=200,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=(dtype == torch.bfloat16),
    fp16=(dtype == torch.float16),
    gradient_checkpointing=True,
    optim="adamw_torch_fused" if dtype == torch.bfloat16 else "adamw_torch",
    weight_decay=0.01,
    max_grad_norm=1.0,
    report_to="none",
    ddp_find_unused_parameters=False,
    remove_unused_columns=False
)

torch.backends.cuda.matmul.allow_tf32 = True

In [11]:
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to:", OUTPUT_DIR)

/tmp/ipykernel_20/1456092904.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
200,0.797200
400,0.299800
600,0.223900
800,0.189500
1000,0.148300
1200,0.086100
1400,0.079000
1600,0.068700
1800,0.063800


Saved to: /kaggle/working/qwen2.5-1_5b-gencls-lora
